# Dynamic programming (DP)

Idea of breaking down a larger problems recursively into subproblems

Within context of RL, it essentially means turning the Bellman equation into an update rule and computing action-value functions using this.

Requires a perfect model of env.

All methods in RL are approximations of DP


##### Policy evaluation (Prediction)

How to compute action-value function

From previous section we know:

<img src="../image-66.png" width="500px" /><div/>


State-value function is the average of all the available action-values, weighted by how likely we are to choose them under our current policy:

We apply the Bellmann equation as an update rule, and compute for each state

<img src="../image-67.png" width="500px" /><div/>



<img src="../image-68.png" width="500px" /><div/>

###### Code implementation

Bellman equation to update V: 

$V(s)\leftarrow r(s)+\gamma\sum_{s'}P(s'|s)V(s')$

In [ ]:
def iterative_policy_evaluation(
    states,
    transitions,
    rewards,
    terminal_states,
    gamma=1.0,
    theta=1e-10,
    max_iters=100_000,
):
    """Iterative policy evaluation for V under a fixed (implicit) policy.

    Here, the transition model already reflects the policy (MRP-style), so we evaluate:
    V(s) = r(s) + gamma * sum_{s'} P(s'|s) * V(s')
    """
    V = {s: 0.0 for s in states}

    # Keep terminal values fixed (single terminal reward, then stop).
    for t in terminal_states:
        V[t] = float(rewards[t])

    deltas = []

    for i in range(max_iters):
        delta = 0.0

        for s in states:
            if s in terminal_states:
                continue

            v_old = V[s]
            exp_next = sum(p * V[s_next] for p, s_next in transitions[s])
            V[s] = float(rewards[s]) + gamma * exp_next
            delta = max(delta, abs(v_old - V[s]))

        deltas.append(delta)
        if delta < theta:
            print(f"Converged after {i + 1} iterations (delta={delta:.3e})")
            break

    return V, deltas


V_eval, deltas = iterative_policy_evaluation(
    states=states,
    transitions=transitions_with_terminal_states,
    rewards=r,
    terminal_states=terminal_states,
    gamma=1.0,
    theta=1e-10,
)

print("\nEstimated V under the current policy:")
for s in states:
    print(f"  {s:>10}: {V_eval[s]: .6f}")
print("\nAnalytical Solution:")
for state in range(len(states)):
    print(f"  {states[state]:>10}: {analytical_solution[state]: .6f}")

Converged after 479 iterations (delta=9.637e-11)

Estimated V under the current policy:
      Class1: -12.543210
      Class2:  1.456790
      Class3:  4.320988
    Facebook: -22.543210
         Pub:  0.802469
       Sleep:  0.000000
        Pass:  10.000000

Analytical Solution:
      Class1: -12.543210
      Class2:  1.456790
      Class3:  4.320988
    Facebook:  10.000000
         Pub:  0.802469
       Sleep: -22.543210
        Pass:  0.000000


#### Policy improvement

From predicting value (estimation) -> move to control problem



Similarly, the action-value function is the average value of the states the agent might transition to given the chosen action, but weighted instead by their transition probabilities

<img src="../image-69.png" width="500px" /><div/>

<img src="../image-44.png" width="600px" /><div/>

We do a one step policy improvement

#### Backup diagram

<img src="../image-75.png" width="600px" /><div/>




So action-value is the immediate reward for taking that action, plus the expected value of the next state, over all possible successor states

If this is better than the value of π, we are better off changing our policy in such a way. And we can indeed answer such question via the policy improvement theorem, which states that if, for all states s:

<img src="../image-70.png" width="200px" /><div/>

The improved policy is better:

<img src="../image-71.png" width="200px" /><div/>




### Policy iteration

chain these improvement steps with policy evaluation


<img src="../image-72.png" width="700px" /><div/>

Each of the resulting policies are monotonically improving, and — since finite MDPs only have finite states — this process must converge to the optimal policy eventually!

So first algorithm for solving an RL problem — called policy iteration


<img src="../image-73.png" width="700px" /><div/>
